In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
import time
import gc

# =====================================================================
# Constrained 16-bit Kapur maximum-entropy threshold segmentation
# =====================================================================
def threshold_kapur_uint16_constrained(image_uint16, max_pore_percent=0.15):
    """
    Kapur thresholding with a constrained maximum-entropy search.
    max_pore_percent: estimated upper porosity bound for the sandstone batch (default 0.15).
    Search within this cumulative-probability interval to constrain the threshold for tight sandstone.
    """
    # Calculate the global histogram over [0,65535]
    hist = np.bincount(image_uint16.ravel(), minlength=65536)
    
    # Calculate the probability and cumulative distributions
    p = hist / float(hist.sum())
    cdf = p.cumsum()
    
    # Select nonzero probabilities whose cumulative probability is below the porosity bound
    valid_idx = (cdf > 0) & (cdf < max_pore_percent)
    
    p_safe = np.copy(p)
    p_safe[p_safe == 0] = 1e-10
    
    # Precompute cumulative entropy terms
    entropy_term = p * np.log(p_safe)
    H_T = np.cumsum(entropy_term)
    H_total = H_T[-1]
    
    total_entropy = np.zeros(65536)
    
    # Calculate background and foreground entropy using vectorized operations
    P = cdf[valid_idx]
    P_comp = 1.0 - P
    
    H_L = - (H_T[valid_idx] / P) + np.log(P)
    H_H = - ((H_total - H_T[valid_idx]) / P_comp) + np.log(P_comp)
    
    total_entropy[valid_idx] = H_L + H_H
    
    # Return the intensity index that maximizes total entropy
    best_t = np.argmax(total_entropy)
    return best_t

# =====================================================================
# 1. Data directories and processing parameters
# =====================================================================
input_dir = '../../data/core_images_256'       # Source RAW directory
output_dir = '../../outputs/segmented_binary_256'    # Output directory

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Volume dimensions
nx, ny, nz = 256, 256, 256

# NLM parameters used for comparison with Avizo
h = 0.035
search_radius = 3
patch_size = 5
h_squared = h ** 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device: {device}")
print("=====================================================================\n")

# =====================================================================
# 2. Process the volume batch
# =====================================================================
file_list = glob.glob(os.path.join(input_dir, '*.raw'))
print(f"Found {len(file_list)} RAW files; starting batch processing...\n")

for i, filepath in enumerate(file_list, 1):
    filename = os.path.basename(filepath)
    name_without_ext = os.path.splitext(filename)[0]
    
    print(f"[{i}/{len(file_list)}] Processing: {filename} ...")
    start_time = time.time()
    
    # --- A. Read the source 16-bit little-endian data ---
    try:
        raw_data = np.fromfile(filepath, dtype='<u2')
        if raw_data.size != nx * ny * nz:
            print(f"  -> [Warning] {filename}: shape mismatch; skipped.")
            continue
        data_3d = raw_data.reshape((nz, ny, nx))
    except Exception as e:
        print(f"  -> [Error] Failed to read {filename}: {e}; skipped.")
        continue

    # --- B. GPU NLM filtering ---
    # Normalize using the 99.9th percentile to limit extreme bright artifacts
    data_float = data_3d.astype(np.float32)
    p_99_9 = np.percentile(data_float, 99.9) 
    
    data_float = np.clip(data_float, 0.0, p_99_9)
    if p_99_9 > 0:
        data_float /= p_99_9

    img_tensor = torch.from_numpy(data_float).unsqueeze(0).unsqueeze(0).to(device)
    pad_tuple = (search_radius, search_radius, search_radius, search_radius, search_radius, search_radius)
    padded_img = F.pad(img_tensor, pad_tuple, mode='replicate')

    output_tensor = torch.zeros_like(img_tensor)
    weight_sum = torch.zeros_like(img_tensor)

    # Compute shifted 3D convolution responses
    for dz in range(-search_radius, search_radius + 1):
        for dy in range(-search_radius, search_radius + 1):
            for dx in range(-search_radius, search_radius + 1):
                z_start = search_radius + dz
                y_start = search_radius + dy
                x_start = search_radius + dx
                shifted_img = padded_img[:, :, z_start:z_start+nz, y_start:y_start+ny, x_start:x_start+nx]
                
                sq_diff = (img_tensor - shifted_img) ** 2
                patch_sq_diff = F.avg_pool3d(sq_diff, kernel_size=patch_size, stride=1, padding=patch_size//2)
                weights = torch.exp(-patch_sq_diff / h_squared)
                
                output_tensor += weights * shifted_img
                weight_sum += weights

    # Move the filtered result to CPU
    denoised_tensor = output_tensor / weight_sum
    denoised_data = denoised_tensor.squeeze().cpu().numpy()

    # --- C. Release GPU memory between volumes ---
    del img_tensor, padded_img, output_tensor, weight_sum, denoised_tensor
    del shifted_img, sq_diff, patch_sq_diff, weights
    gc.collect()
    torch.cuda.empty_cache()

    # --- D. Restore 16-bit intensities and apply constrained Kapur thresholding ---
    # Clip the range and restore grayscale intensities to [0,65535]
    denoised_data = np.clip(denoised_data, 0.0, 1.0)
    filtered_data_uint16 = (denoised_data * 65535).astype(np.uint16)
    
    # Calculate the uint16 threshold using constrained Kapur thresholding
    # For very low porosity, consider a lower max_pore_percent such as 0.10 or 0.05
    threshold_value = threshold_kapur_uint16_constrained(filtered_data_uint16, max_pore_percent=0.15)
    
    # Binarize the pore phase: intensities below the threshold are pores
    binary_3d = filtered_data_uint16 < threshold_value
    
    # Calculate porosity for the current volume
    pore_voxels = np.sum(binary_3d)
    porosity = pore_voxels / binary_3d.size

    # --- E. Name and save outputs with a porosity label ---
    # Include porosity to four decimal places in the filename
    cond_label = f"{porosity:.4f}"
    
    filtered_filepath = os.path.join(output_dir, f"{name_without_ext}_cond_{cond_label}_filtered.raw")
    output_filepath = os.path.join(output_dir, f"{name_without_ext}_cond_{cond_label}_binary.raw")

    # Save filtered 16-bit grayscale data in little-endian format for Avizo
    filtered_data_uint16.astype('<u2').tofile(filtered_filepath)

    # Save the binary volume as uint8 with values 0 and 255
    output_data = (binary_3d * 255).astype(np.uint8) 
    output_data.tofile(output_filepath)
    
    print(f"  -> Complete! Time: {time.time() - start_time:.2f}s | Threshold: {threshold_value} | Porosity: {porosity * 100:.2f}%")

print("\nAll files have been processed and labeled.")